# PA005 - High Value Customer Identification (Insiders)

## SOLUTION PLANNING

"All in one place" is an online retail store that sells second-hand products from various brands at lower prices.

With just over a year of operation, the marketing team noticed that some customers from their base purchase more expensive products more frequently, contributing significantly to the company's revenue.

Based on this insight, the marketing team decided to launch a loyalty program for the top customers in their base, named Insiders. However, the marketing team lacks the necessary knowledge to select the participating customers for this program.

As a result, this task was assigned to the company's data team, where a solution that ultimately provides a list of customers to be invited to participate in the Insiders program should be developed. Additionally, a report answering the following questions should be delivered:

1. Who are the eligible individuals to participate in the program?
   - Faturamento:
        - Alto ticket medio
        - Alto LTV
        - Baixa recencia
        - Baixa probabilidade de churn
        - Alta previsao LTV
        - Alta propensao de compra

    - Custo:
        - Baixa taxa de devolucao

   - Experiencia de Compra
        - Media alta das avaliacoes    

2. How many customers will be part of the group?
   - Numero total de clientes
   - % do grupo insiders
    
3.  What are the main characteristics of these customers?
    - Escrever caracteristicas do cliente:
      - Idade
      - Localizacao
       
    - Escrever caracteristicas do consumo:
      - Atributos da clusterizacao

4. What percentage of the revenue comes from the selected group?
    - Faturamento total do ano
    - Faturamento do grupo Insiders

5. What is the revenue expectation for this group in the upcoming months?
    - LTV do grupo Insiders
    - Analise de Cohort
    
6. What are the conditions for someone to be eligible for the group?
   - Definir a periodicidade ( 1 month, 2 month, etc.). 
   - A pessoa precisa ser similar com uma outra pessoa do grupo.

7. What are the conditions for someone to be removed from the group?
   - Definir a periodicidade ( 1 month, 2 month, etc.). 
   - A pessoa precisa ser desimilar com uma outra pessoa do grupo.
   
8. What assurance is there that the group is better than the rest of the base?
   - Teste A/B.
   - Teste A/B Bayesiano.
   - Teste de hipotese.

9. What actions can the marketing team take to increase revenue?
   - Desconto
   - Preferencia de Compra
   - Visitar empresa

**SOLUTION PLANNING**

1 - Input:
    Business Problem: Hight value customers identification and selection in order to join a loyalty program.
    Dataset: sales datas from an e-commerce, collected trough an one year period.

2 - Output:
    List containing the customer identification and if they are eligible or not to the loyalty program.
    Report, answering the business questions and explaining how the selection was made.
    
3 - Tasks
    - Who are the eligible individuals to participate in the program?
        Understanding of what elegible means and what are the most valuable customer.
        
        
    - How many customers will be part of the group?
        
        
    - What are the main characteristics of these customers?
        
    
    - What percentage of the revenue comes from the selected group?
        
    
    - What is the revenue expectation for this group in the upcoming months?
        
    
    - What are the conditions for someone to be eligible for the group?
        
    
    - What are the conditions for someone to be removed from the group?
        
    
    - What assurance is there that the group is better than the rest of the base?
        
    
    - What actions can the marketing team take to increase revenue?
        


# 0 - IMPORTS </font>

In [1]:
# Data Maniputalion and Data Analysis
import re
import pandas  as pd
import numpy   as np
import seaborn as sns

from matplotlib import pyplot  as plt
from plotly     import express as px

from sklearn import preprocessing as pp

# ML Algorithms
import umap.umap_ as umap

from sklearn import cluster as c
from sklearn import metrics as mt
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

# Loading Images
from IPython.display         import Image, display
from IPython.core.display    import HTML

C:\Users\perot\anaconda3\envs\pa_clustering\lib\site-packages\umap\distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
C:\Users\perot\anaconda3\envs\pa_clustering\lib\site-packages\umap\distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
C:\Users\perot\anaconda3\envs\pa_clustering\lib\site-packages\umap\distanc

## 0.1 - Helper Functions

In [ ]:
def jupyter_settings():
    %matplotlib inline
    %pylab inline
    
    plt.style.use( 'bmh' )
    plt.rcParams['figure.figsize'] = [25,12]
    plt.rcParams['font.size'] = 24
    
    display( HTML( '<style>.container {width:100% !important; }</style>') )
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.set_option( 'display.expand_frame_repr', False )
    
    sns.set()
jupyter_settings()

# 1 - DATA LOADING

## 1.1 - Loading the Datas

In [2]:
df = pd.read_csv("../data/raw/Ecommerce.csv")

# drop extra column

df = df.drop( columns=['Unnamed: 8'], axis=1)


In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,29-Nov-16,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,29-Nov-16,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,29-Nov-16,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,29-Nov-16,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,29-Nov-16,3.39,17850.0,United Kingdom


## 1.2 - DATA DESCRIPTIVE

### 1.2.1 - RENAME COLUMNS

In [4]:
df1 = df.copy()

In [5]:
cols_new = ['invoice_no','stock_code','description','quantity','invoice_date','unit_price','customer_id','country']
df1.columns = cols_new

### 1.2.2 - DATA DIMENSIONS

In [6]:
print('Number of columns: {}'.format( df1.shape[1] ) )
print('Number of rows: {}'.format( df1.shape[0] ) )

Number of columns: 8
Number of rows: 541909


### 1.2.3 - DATA TYPES

In [7]:
df1.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

### 1.2.4 - CHECK NA VALUES

In [8]:
df1.isna().sum()

invoice_no           0
stock_code           0
description       1454
quantity             0
invoice_date         0
unit_price           0
customer_id     135080
country              0
dtype: int64

### 1.2.5 - REPLACE NA VALUES

In [9]:
# Remove NA
df1 = df1.dropna( subset=['customer_id', 'customer_id'] )
print( 'Removed data"{:2f}%'.format( 1 - (df1.shape[0] / df.shape[0] ) ) )

Removed data"0.249267%


### 1.2.6 - CHANGE DTYPES

In [10]:
df1.dtypes

invoice_no       object
stock_code       object
description      object
quantity          int64
invoice_date     object
unit_price      float64
customer_id     float64
country          object
dtype: object

In [11]:
df1[ 'invoice_date'] = pd.to_datetime( df1['invoice_date'], format='%d-%b-%y')

df1['customer_id'] = df1['customer_id'].astype( int )

In [12]:
 df1.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2016-11-29,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2016-11-29,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2016-11-29,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2016-11-29,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2016-11-29,3.39,17850,United Kingdom


### 1.2.7 - DESCRIPTIVE STATISTICS

In [13]:
num_attributes = df1.select_dtypes( include=['int64', 'float64'] )
cat_attributes = df1.select_dtypes( exclude=['int64', 'float64', 'datetime64[ns]'] )

### 1.2.7.1 - Numerical Attributes

In [14]:
# Central Tendency
ct1 = pd.DataFrame( num_attributes.apply( np.mean ) ).T
ct2 = pd.DataFrame( num_attributes.apply( np.median ) ).T

# Dispersion - 
d1 = pd.DataFrame( num_attributes.apply( np.std ) ).T
d2 = pd.DataFrame( num_attributes.apply( np.min ) ).T
d3 = pd.DataFrame( num_attributes.apply( np.max ) ).T
d4 = pd.DataFrame( num_attributes.apply( lambda x: x.max() - x.min() ) ).T
d5 = pd.DataFrame( num_attributes.apply( lambda x: x.skew() ) ).T
d6 = pd.DataFrame( num_attributes.apply( lambda x: x.kurtosis() ) ).T

# Concatenate
m = pd.concat( [d2, d3, d4, ct1, ct2, d1, d5, d6]).T.reset_index()
m.columns = ['attributes', 'min', 'max', 'range', 'mean', 'median', 'std', 'skew', 'kurtosis']
m

,attributes,min,max,range,mean,median,std,skew,kurtosis
0,quantity,-80995.0,80995.0,161990.0,12.061303,5.00,248.693064,0.182663,94317.563673
1,unit_price,0.0,38970.0,38970.0,3.460471,1.95,69.315077,452.219019,246924.542988


#### 1.2.7.1 - Numerical Attributes Investigate 

In [15]:
quantity = negative quantity
unit_price = price equal 0

SyntaxError: invalid syntax (2039558827.py, line 1)

### 1.2.7.2 - Categorical Attributes

##### Invoice No

In [16]:
# Problem: There are letters and numbers on the invoice_no column
df1.loc[df1['invoice_no'].apply( lambda x: bool( re.search('[^0-9]+', x) ) ), :]

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
141,C536379,D,Discount,-1,2016-11-29,27.50,14527,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2016-11-29,4.65,15311,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2016-11-29,1.65,17548,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2016-11-29,0.29,17548,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2016-11-29,0.29,17548,United Kingdom
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2017-12-07,0.83,14397,United Kingdom
541541,C581499,M,Manual,-1,2017-12-07,224.69,15498,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2017-12-07,10.95,15311,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2017-12-07,1.25,17315,United Kingdom


In [17]:
# SOLUTION: Understand if every invoice_no that contais letter are negative and decide if these will be removed, separated, etc.

# How to find out if every invoice_nmo that starts with "C" has a negative quantity

# Total of invoice_no that contais letters
aux1 = df1.loc[df1['invoice_no'].apply( lambda x: bool( re.search('[^0-9]+', x) ) ), :]
len(aux1)

# Qty of invoices_no that the qty is negative
len( aux1[aux1['quantity']<0])

8905

##### Stock Code

In [18]:
# Problem: There are codes on stock_code that are composed of letters and we do not know the meaning
df1.loc[df1['stock_code'].apply( lambda x: bool( re.search('^[a-zA-Z]+$', x ) ) ), 'stock_code'].unique()

# SOLUTION: Revoce sotck_code composed of letters or codes as per ['POST', 'D', 'M', 'PADS', 'DOT', 'CRUK']


array(['POST', 'D', 'M', 'PADS', 'DOT', 'CRUK'], dtype=object)

##### Description

In [ ]:
# ACTION: REMOVE THIS COLUMN FROM THE DATASET AS THIS DO NOT HELPS TO GENERATE A CLUSTER
# df1.head()

##### Country

In [19]:
df1['country'].unique()

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Greece', 'Singapore', 'Lebanon',
       'United Arab Emirates', 'Saudi Arabia', 'Czech Republic', 'Canada',
       'Unspecified', 'Brazil', 'USA', 'European Community', 'Bahrain',
       'Malta', 'RSA'], dtype=object)

In [20]:
# Qty sales made per country
df1['country'].value_counts( normalize=True )

United Kingdom          0.889509
Germany                 0.023339
France                  0.020871
EIRE                    0.018398
Spain                   0.006226
Netherlands             0.005828
Belgium                 0.005086
Switzerland             0.004614
Portugal                0.003638
Australia               0.003095
Norway                  0.002669
Italy                   0.001974
Channel Islands         0.001863
Finland                 0.001708
Cyprus                  0.001529
Sweden                  0.001136
Austria                 0.000986
Denmark                 0.000956
Japan                   0.000880
Poland                  0.000838
USA                     0.000715
Israel                  0.000615
Unspecified             0.000600
Singapore               0.000563
Iceland                 0.000447
Canada                  0.000371
Greece                  0.000359
Malta                   0.000312
United Arab Emirates    0.000167
European Community      0.000150
RSA       

# Cycle 2: 2 - DATA FILTERING

In [ ]:
On this project, the step DATA FILTERING will be done earlier as some features are been calculated from the dataset and are combined with that dataset later on, and it's our duty to make sure that we are keeping our data clean and without errors.

In [ ]:
df2 = df1.copy()

In [ ]:
# --- NUMERICAL ATTRIBUTES ---
df2 = df2.loc[df2['unit_price'] >= 0.04, :]

# --- CATEGORICAL ATTRIBUTES ---
df2 = df2[~df2['stock_code'].isin( ['POST', 'D', 'M', 'PADS', 'DOT', 'CRUK'] ) ]

# Description
df2 = df2.drop( columns='description', axis=1)

# MAP
df2 = df2[~df2['country']. isin( ['European Community','Unspecified'] ) ]

# Quantity
df2_returns = df2.loc[df1['quantity'] < 0, :]
df2_purchases = df2.loc[df1['quantity'] >=0, :]

# 3 - FATURE ENGINEERING

In [ ]:
# data reference
df_ref = df2.drop( ['invoice_no','stock_code','quantity','invoice_date','unit_price','country'],axis=1).drop_duplicates(ignore_index=True)

In [ ]:
# Gross Revenue
df2_purchases.loc[:, 'gross_revenue'] = df2_purchases.loc[:, 'quantity'] * df2_purchases.loc[:, 'unit_price']

# Monetary
df_monetary = df2_purchases[['customer_id', 'gross_revenue']].groupby('customer_id').sum().reset_index()
df_ref = pd.merge( df_ref, df_monetary, on='customer_id', how='left')

In [ ]:
# Recency
df_recency = df2_purchases[['customer_id', 'invoice_date']].groupby('customer_id').max().reset_index()
df_recency['recency_days'] = (df2_purchases['invoice_date'].max() - df_recency['invoice_date']).dt.days
df_recency = df_recency[['customer_id','recency_days']].copy()
df_ref = pd.merge( df_ref, df_recency, on='customer_id', how='left')

In [ ]:
# Frequency
df_freq = df2_purchases.loc[:, ['customer_id', 'invoice_no']].drop_duplicates().groupby('customer_id').count().reset_index()
df_ref = pd.merge(df_ref, df_freq, on='customer_id', how='left')

In [ ]:
# Avg Ticket
df_avg_ticket = df2_purchases.loc[:, ['customer_id','gross_revenue']].groupby('customer_id').mean().reset_index().rename( columns={'gross_revenue':'avg_ticket'})
df_ref = pd.merge( df_ref, df_avg_ticket, on='customer_id', how='left')

In [ ]:
df_ref.isna().sum()

# <font color='red'> 4 - EDA </font>

In [ ]:
df4 = df_ref.dropna()

# <font color='red'> 5 - DATA PREPARATION </font>

In [ ]:
df5 = df4.copy()

In [ ]:
# Standard Scaler
ss = pp.StandardScaler()

df5['gross_revenue'] = ss.fit_transform( df5[['gross_revenue']] )
df5['recency_days'] = ss.fit_transform( df5[['recency_days']] )
df5['invoice_no'] = ss.fit_transform( df5[['invoice_no']] )
df5['avg_ticket'] = ss.fit_transform( df5[['avg_ticket']] )

# <font color='red'> 6 - FEATURE SELECTION </font>

In [ ]:
df6 = df5.copy()

# 7 - FINE TUNING

In [ ]:
X = df6.drop( columns=['customer_id'])

In [ ]:
cluster = [2,3,4,5,6,7]

## 7.1 -  K-Means within-cluster sum of square (WSS)

In [ ]:
# O K-Means vai definir, aleatoriamente, n pontos centrais no meu conjunto de dados. A quantidade de pontos e definida pelo usuario, como sendo o numero de cluster desejado.
# A partir da definicao dos pontos centrais, o K-Means vai atribuir cada ponto ao centroide mais proximo, calculando a distancia euclidiana entre o ponto e o centroide.
# Para cada grupo formado, o algoritmo vai recalcular o centroide como sendo a media de todos os pontos pertencentes aquele grupo. Movendo o centroide para uma posicao que minimiza a soma dos quadrados das distancias. E esse processo se 
# repete ate que os centroides nao tenham uma mudanca significativa em suas posicoes.
# Para a Metrica WSS:
    # O K-Means vai calcular a distancia entre cada um dos pontos em relacao ao ponto central para cada um dos clusters, somar esses valores, gerando um valor total para cada cluster. O WSS eh dado somando esse valor total para todos 
    # clusters. E entende-se que o numero de cluster ideal eh aquele cujo apresenta uma diferenca maior no valor da metrica WSS.
# Esta metrica nao leva em consideracao a distancia entre os clusters, o que pode vir a ser um problema, uma vez que podemos ter clusters muito proximos, o que torna dificil a separacao.

In [ ]:

# URL da imagem
url_imagem = '../reports/figures/wss.png'  # Substitua pela URL da sua imagem

# Exibe a imagem no notebook
display(Image(url=url_imagem))

In [ ]:
wss = []
for k in cluster:
    # Model Definition
    kmeans = c.KMeans(init='random', n_clusters=k, n_init=10, max_iter=300, random_state=42)
    
    # Model Training
    kmeans.fit( X )

    # Validation
    wss.append( kmeans.inertia_ ) 

# plot wss
plt.plot( cluster, wss, linestyle='--', marker='o', color='b'); 
plt.xlabel('Clusters'); 
plt.ylabel('Within-Cluster Sum of Square'); 
plt.title('WSS vs K'); 

In [ ]:
kmeans = KElbowVisualizer( c.KMeans(), k=cluster, timings=False );
kmeans.fit( X );
kmeans.show();

## 7.2 -  Silhouette Score

In [ ]:
#O Silhouette Score é uma métrica de avaliação de clusters que mede o quão bem os pontos de dados estão agrupados. Ele leva em consideração a distância média entre os pontos dentro do mesmo cluster (a) e a distância média entre os pontos de um cluster para o cluster
#mais próximo diferente dele (b). Esses valores são então usados para calcular um valor de silhueta para cada ponto de dados.

#URL da imagem
url_imagem = '../reports/figures/ss_2.png'  # Substitua pela URL da sua imagem

# Exibe a imagem no notebook
display(Image(url=url_imagem))

In [ ]:
# O valor de S(i) varia de -1 a 1. Quanto mais próximo o valor estiver de 1, melhor o ponto está agrupado. Se o valor for próximo a 0, significa que o ponto está próximo à fronteira entre dois clusters. Se o valor for negativo, indica que o ponto pode ter sido atribuído ao 
# cluster errado.

# O Silhouette Score global para todos os pontos de dados é a média dos valores de silhueta individuais para todos os pontos:
# URL da imagem
url_imagem = '../reports/figures/ss_3.png'  # Substitua pela URL da sua imagem

# Exibe a imagem no notebook
display(Image(url=url_imagem))

In [ ]:
kmeans = KElbowVisualizer( c.KMeans(), k=cluster, metric='silhouette', timings=False );
kmeans.fit( X );
kmeans.show();

## 7.3 Silhouette Analysis

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(25, 18))

for k in cluster:
    km = c.KMeans( n_clusters=k, init='random', n_init=10, max_iter=100, random_state=42 )
    q, mod = divmod(k, 2)
    visualizer = SilhouetteVisualizer( km, color='yellowbrick', ax=ax[q-1][mod])
    visualizer.fit( X )
    visualizer.finalize()


# 8 - MACHINE LEARNING TRAINING

In [ ]:
# Model Definition
k = 5
kmeans = c.KMeans( init='random', n_clusters=k, n_init=10, max_iter=300, random_state=42 )

# Model Training
kmeans.fit( X )

# Model Prediction
labels = kmeans.labels_

## 8.1 - Cluster Validation

In [ ]:
# WSS
print('WSS value: {}'. format( kmeans.inertia_) )

# Silhouette Score
print( 'SS Score: {}'.format( mt.silhouette_score( X, labels, metric='euclidean' ) ) )

# 9 - CLUSTER ANALYSIS

In [ ]:
df9 = df6.copy()
df9['cluster'] = labels
df9.head()

## 9.1 - Vizualization Inspection


In [ ]:
## Silhouette 4 clusters
visualizer = SilhouetteVisualizer( kmeans, colors='yellowbrick' )
visualizer.fit( X )
visualizer.finalize()

## 9.2 - 2D Plot

In [ ]:
df9.head()

In [ ]:
df_viz = df9.drop( columns='customer_id', axis=1)
sns.pairplot( df_viz, hue='cluster')

## 9.3 - UMAP

In [ ]:
Machine Learning - Manifold

Aprendizado por topologia: PCA - Metodo baseado em matriz ou espaco entre distancias. Temos 9 condicoes, cumprir 9 colorarios para ter uma garantia de espaco. Espaco de Hilbert, etc.AffinityPropagation

UMAP, T-SNE ( 2009 )- Abordagem por topologia ( Manifold ). Topologia sao grafos em alta dimensionalidade

In [ ]:
# UMAP 
reducer = umap.UMAP( n_neighbors=20, random_state=42 )
embedding = reducer.fit_transform( X )

# Embedding 
df_viz['embedding_x'] = embedding[:, 0]
df_viz['embedding_y'] = embedding[:, 1]

# Plot UMAP
sns.scatterplot( 
                 x='embedding_x',
                 y='embedding_y', 
                 hue='cluster',
                 palette=sns.color_palette('hls',
                                           n_colors=len(df_viz['cluster'].unique())),
                 data=df_viz)
                                           


## 9.4 - Cluster Profile

In [ ]:
# Number of Customer
df_cluster = df9[['customer_id', 'cluster']].groupby('cluster').count().reset_index()
df_cluster['perc_customer'] = 100*( df_cluster['customer_id'] / df_cluster['customer_id'].sum() )

# Avg Gross Revenue
df_avg_gross_revenue = df9[['gross_revenue', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_gross_revenue, how='inner', on='cluster')

# Avg Recency Days
df_avg_recency_days = df9[['recency_days', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_recency_days, how='inner', on='cluster')

# Avg invoice_no
df_invoice_no = df9[['invoice_no', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_invoice_no, how='inner', on='cluster')

# Avg Ticket
df_avg_ticket = df9[['avg_ticket', 'cluster']].groupby('cluster').mean().reset_index()
df_cluster = pd.merge( df_cluster, df_avg_ticket, how='inner', on='cluster')


In [ ]:
df_cluster.head()

# 10 - DEPOLOY